In [30]:
from helpers import *
import cdd

def generator_filter(v, r, r0, eps=1e-9):
    print(v)
    sm = sum(r[i]*v[i] for i in range(len(v)))
    print(v, r, r0, sm, sm - r0, sm - r0 <= 0)
    return sm - r0 == 0

def generate_zk(phifk, Gk):
    # print(f"Fk-1: {F_k_minus_1}")
    # print(f"Gk: {Gk}")
    L = len(phifk)
    row, r = Gk.eqs[0]
    print("row", row)

    #constraints_geq_0 = [([0] + [1 if v == w else 0 for w in range(L)]) for v in range(L) if row[v] == 0] # ???
    constraints_geq_0 = [([0] + [1 if v == w else 0 for w in range(L)]) for v in range(L)] # ???
    constraints_leq_1 = [([1] + [-1 if v == w else 0 for w in range(L)]) for v in range(L)]
    #constraints_geq_0 = []
    #constraints_leq_1 = []
    constraints_eqs = [[r] + [-x for x in row]]
    #constraints_eqs = []
    
    mat = cdd.matrix_from_array(constraints_geq_0 + constraints_leq_1 + constraints_eqs)
    # print(mat)
    mat.rep_type = cdd.RepType.INEQUALITY
    poly = cdd.polyhedron_from_matrix(mat)
    generators_raw = cdd.copy_generators(poly).array
    print(f"Gens raw: {generators_raw}")
    # print("d[0]", [d[0] for d in generators_raw])
    generators = [[Frac(x).limit_denominator(1000) for x in d[1:]] for d in generators_raw 
                  if generator_filter(d[1:], row, r)]
    generators_new = []
    for g in generators:
        add = True
        for v in range(L):
            if g[v] == 0 and row[v] != 0:
                add = False
        if add:
            generators_new.append(g)
    print(f"Gens:", [str_list(g) for g in generators])
    print(f"Gens new:", [str_list(g) for g in generators_new])
    print(f"Phi(F_k_minus) {phifk}")
    Zk = [d for d in generators_new if vector_leq(phifk, d)]
    print("Zk", [str_list(g) for g in Zk])
    return Zk

phifk = [0,0]
Gk = LowerSet([([1,1], 0.5)])
res = generate_zk(phifk, Gk) 
print()
phifk = [0,0]
Gk = LowerSet([([1,0], 0.5)])
res = generate_zk(phifk, Gk) 
print()
phifk = [0,0]
Gk = LowerSet([([0,1], 0.5)])
res = generate_zk(phifk, Gk) 

row [1, 1]
Gens raw: [[1.0, 0.0, 0.5], [1.0, 0.5, 0.0], [1.0, 0.0, 0.0]]
[0.0, 0.5]
[0.0, 0.5] [1, 1] 0.5 0.5 0.0 True
[0.5, 0.0]
[0.5, 0.0] [1, 1] 0.5 0.5 0.0 True
[0.0, 0.0]
[0.0, 0.0] [1, 1] 0.5 0.0 -0.5 True
Gens: ['[0, 1/2]', '[1/2, 0]']
Gens new: []
Phi(F_k_minus) [0, 0]
Zk []

row [1, 0]
Gens raw: [[1.0, 0.0, 0.0], [1.0, 0.5, 0.0], [1.0, 0.0, 1.0], [1.0, 0.5, 1.0]]
[0.0, 0.0]
[0.0, 0.0] [1, 0] 0.5 0.0 -0.5 True
[0.5, 0.0]
[0.5, 0.0] [1, 0] 0.5 0.5 0.0 True
[0.0, 1.0]
[0.0, 1.0] [1, 0] 0.5 0.0 -0.5 True
[0.5, 1.0]
[0.5, 1.0] [1, 0] 0.5 0.5 0.0 True
Gens: ['[1/2, 0]', '[1/2, 1]']
Gens new: ['[1/2, 0]', '[1/2, 1]']
Phi(F_k_minus) [0, 0]
Zk ['[1/2, 0]', '[1/2, 1]']

row [0, 1]
Gens raw: [[1.0, 0.0, 0.5], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0], [1.0, 1.0, 0.5]]
[0.0, 0.5]
[0.0, 0.5] [0, 1] 0.5 0.5 0.0 True
[0.0, 0.0]
[0.0, 0.0] [0, 1] 0.5 0.0 -0.5 True
[1.0, 0.0]
[1.0, 0.0] [0, 1] 0.5 0.0 -0.5 True
[1.0, 0.5]
[1.0, 0.5] [0, 1] 0.5 0.5 0.0 True
Gens: ['[0, 1/2]', '[1, 1/2]']
Gens new: ['[0,

In [23]:
import numpy as np
import matplotlib.pyplot as plt

def plot_halfspace(H):
    # grid for visualization
    x = np.linspace(-2, 2, 400)
    y = np.linspace(-2, 2, 400)
    X, Y = np.meshgrid(x, y)
    
    # check feasibility of each point
    feasible = np.ones_like(X, dtype=bool)
    
    for b, a1, a2 in H:
        feasible &= (b + a1*X + a2*Y >= 0)
    
    plt.figure(figsize=(6,6))
    
    # shade feasible region
    plt.contourf(X, Y, feasible, levels=[0.5,1], alpha=0.3)
    
    # draw boundary lines
    for b, a1, a2 in H:
        if abs(a2) > 1e-8:
            y_line = (-b - a1*x)/a2
            plt.plot(x, y_line)
        else:
            if a1 == 0:
                a1 = 1/10**9
            x_line = -b/a1
            plt.axvline(x_line)
    
    plt.axhline(0)
    plt.axvline(0)
    
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title("Halfspace intersection from cdd output")
    plt.xlim(-2,2)
    plt.ylim(-2,2)
    plt.gca().set_aspect('equal')
    
    plt.show()

In [14]:
def is_binary(x, eps=1e-9):
    return abs(x) < eps or abs(x-1) < eps

